In [22]:
import pandas as pd
import numpy as np
import re

import tmdbsimple as tmdb

## Завантаження наших даних та даних з imdb

In [302]:
df = pd.read_excel('backup.xlsx')

In [271]:
df = df.rename(columns={'ID IMDB': 'tconst'})

In [7]:
#таблиця яка допоможе заповнити пусті imdb коди
akas_imdb =  pd.read_csv('title.akas.tsv', sep='\t', dtype=str, na_values='\\N')

In [8]:
# інші таблиці imdb з яких будемо брати інформацію
basics_imdb = pd.read_csv('title.basics.tsv', sep='\t', dtype=str, na_values='\\N')
crew_imdb = pd.read_csv('title.crew.tsv', sep='\t', dtype=str, na_values='\\N')
ratings_imdb = pd.read_csv('title.ratings.tsv', sep='\t', dtype=str, na_values='\\N')
names_imdb = pd.read_csv('name.basics.tsv', sep='\t', dtype=str, na_values='\\N')

In [9]:
# беремо тільки потрібні колонки
basics_imdb = basics_imdb[['tconst', 'genres']].copy()
crew_imdb = crew_imdb[['tconst', 'directors']].copy()
ratings_imdb = ratings_imdb[['tconst', 'averageRating', 'numVotes']].copy()

In [10]:
akas_imdb = akas_imdb[['titleId', 'title', 'region']].copy()
akas_imdb['title'] = akas_imdb['title'].str.lower().str.strip()

## Заповнення пустих полей з imdb кодами (tconst)

In [272]:
tconst_list = dict()
directors_list = dict()

In [273]:
# функція, яка заповнює усі пусті поля з imdb кодом а також додає цей код в tconst_list,
# щоб кожного разу не ітеруватись по таблиці imdb
def fill_tconst(table):
    for i in range(len(table)):
        current_m = str(table['Фильм'][i])
        if pd.isna(table.at[i, 'tconst']) == True or table['tconst'][i] == '-':
            if current_m in tconst_list:
                table.at[i, 'tconst']  = tconst_list[current_m]
            else:
                code = akas_imdb.loc[akas_imdb['title'] == current_m.lower().strip(), 'titleId']
                tconst_list[current_m] = None
                if not code.empty:
                    table.at[i, 'tconst'] = code.values[0]
                    tconst_list[current_m] = code.values[0]
        else:
            if current_m in tconst_list:
                table.at[i, 'tconst']  = tconst_list[current_m]

In [274]:
fill_tconst(df)

In [275]:
akas_imdb = akas_imdb[akas_imdb['region'] == 'UA'].copy()
df = df.merge(akas_imdb, left_on='tconst', right_on='titleId', how='left')

In [276]:
df = df.merge(basics_imdb, left_on='tconst', right_on='tconst', how='left')

In [277]:
df = df.merge(crew_imdb, left_on='tconst', right_on='tconst', how='left')

In [278]:
df = df.merge(ratings_imdb, left_on='tconst', right_on='tconst', how='left')

In [279]:
#замінюємо код людини на її реальне імʼя використовуючи таблицю names
def director_code_to_name(table):
    for i in range(len(table)):
        name_list = list()
        if isinstance(table['directors'][i], str):
            for person_code in list(table['directors'][i].split(',')):
                if person_code in directors_list:
                    name_list.append(directors_list[person_code])
                else:
                    directors_list[person_code] = None
                    name = names_imdb.loc[names_imdb['nconst'] == person_code.strip(), 'primaryName']
                    if not name.empty:
                        name_list.append(name.values[0])
                        directors_list[person_code] = name.values[0]
            table.at[i, 'directors'] = name_list[0]
        else: table.at[i, 'directors'] = None

In [280]:
director_code_to_name(df)

## Додавання іншої корисної інформацїї з інших ресурсів окрім як imdb

In [281]:
tmdb_dict = dict()

In [282]:
tmdb.API_KEY = 'c10782e59252fd0b99272aa710fff0c3'

def get_tmdb_info(imdb_id):
    try:
        if imdb_id in tmdb_dict: return tmdb_dict[imdb_id]
        result = tmdb.Find(imdb_id).info(external_source='imdb_id')
        movie = result.get('movie_results')
        if not movie:
            tmdb_dict[imdb_id] = [None] * 6
            return tmdb_dict[imdb_id]

        tmdb_id = movie[0]['id']
        movie_data = tmdb.Movies(tmdb_id)
        details = movie_data.info()
        releases = movie_data.release_dates()
        countries = [c['name'] for c in details.get('production_countries', [])][0]
        mpaa = None
        for entry in releases.get('results', []):
            for release in entry['release_dates']:
                if release.get('certification'):
                    mpaa = release['certification']
                    break
            break
        budget = details.get('budget')
        revenue = details.get('revenue')
        original_language = details.get('original_language')
        genres = [g['name'] for g in details.get('genres', [])][0]
        returning_value = (
            countries,
            mpaa,
            budget,
            revenue,
            original_language,
            genres
        )
        tmdb_dict[imdb_id] = returning_value
        return returning_value

    except Exception as e:
        print(f"Error for {imdb_id}: {e}")
        return [None] * 6

In [283]:
# Error бувває якщо tconst не знайдено в датабазі
df[['production_countries', 'mpaa_rating', 'budget', 'revenue', 'original_language', 'genres']] = None

for i, imdb_id in enumerate(df['tconst']):
    if imdb_id in tmdb_dict:
        values = tmdb_dict[imdb_id]
    else: values = get_tmdb_info(imdb_id)
    df.loc[i, ['production_countries', 'mpaa_rating', 'budget', 'revenue', 'original_language', 'genres']] = values

Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range
Error for tt10242898: list index out of range


In [285]:
df['week'] = pd.to_datetime(df['Дата показа']).dt.day_name()

/var/folders/bt/_x0h2pb92g55h0wvnfmyf_900000gp/T/ipykernel_48523/3923950820.py:1: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  df['week'] = pd.to_datetime(df['Дата показа']).dt.day_name()


In [286]:
# де відустій прибуток ставимо нулі
df["Сумма"].fillna(0, inplace=True)
df["Количество"].fillna(0, inplace=True)
df["Количество непроданных"].fillna(0, inplace=True)

In [287]:
#real ticket price value bacause of discounts promocodes etc
df['ticketPriceReal'] = df['Сумма']/df['Количество']

In [288]:
df['averageRating'] = pd.to_numeric(df['averageRating'])
df['numVotes'] = pd.to_numeric(df['numVotes'])

In [289]:
df["averageRating"].fillna(df["averageRating"].mean(), inplace=True)
df["numVotes"].fillna(df["numVotes"].mean(), inplace=True)
df["budget"].fillna(df["budget"].mean(), inplace=True)
df["revenue"].fillna(df["revenue"].mean(), inplace=True)

In [290]:
df = df.drop(columns=['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 4', 'Сеанс', 'Сумма бонусов', 'Сумма сервисный сбор',
                'Сервисный сбор план', 'region', 'titleId', 'tconst', 'title'])

In [291]:
column_translation = {
    'Кинотеатр': 'Cinema',
    'Фильм': 'Film',
    'Продолжительность в минутах': 'durationMin',
    'Дистрибьютор': 'Distributor',
    'Киностудия': 'Studio',
    'Зрительный зал': 'Auditorium',
    'Технология': 'Technology',
    'Номер': 'Number',
    'Сеанс отменен': 'sessionCanceled',
    'Признак премьерного показа': 'isPremiere',
    'Дата показа': 'Date',
    'Время показа': 'Time',
    'Тип посадочного места': 'seatType',
    'Количество': 'quantitySold',
    'Количество план': 'quantityPlan',
    'Количество непроданных': 'quantityUnsold',
    'Сумма': 'profitEarned',
    'Сумма план': 'profitPlan',
    'genres': 'genres',
    'directors': 'directors',
    'averageRating': 'averageRating',
    'numVotes': 'numVotes',
    'production_countries': 'productionCountries',
    'mpaa_rating': 'mpaaRating',
    'budget': 'budget',
    'revenue': 'revenue',
    'original_language': 'originalLanguage',
    'week': 'week'
}



df = df.rename(columns=column_translation)

In [292]:
df['cinemaCity'] = df['Cinema']

df['cinemaCity'] = df['cinemaCity'].replace({
    'NEW Планета-Кіно Львів, Форум': 'Львів',
    'NEW IMAX Київ': 'Київ',
    'NEW Планета Кіно IMAX Одеса,Таїрова': 'Одеса',
    'NEW Планета Кіно Одеса, Котовського': 'Одеса',
    'NEW Планета Кіно Суми': 'Суми',
    'NEW Планета-Кіно IMAX Львів': 'Львів',
    'NEW Планета-Кіно IMAX Харків': 'Харків',
    'Київ (River Mall)': 'Київ',
    'NEW Блокбастер-Кино': 'Київ'

})

df['populationCity'] = df['cinemaCity']

df['populationCity'] = df['populationCity'].replace({
    'Львів': 717273,
    'Київ': 2952000,
    'Одеса': 992874,
    'Суми': 255672,
    'Харків': 1402000

})

df['Cinema'] = df['Cinema'].replace({
    'NEW Планета-Кіно Львів, Форум': 'Cinema 1L',
    'NEW IMAX Київ': 'Cinema 2K',
    'NEW Планета Кіно IMAX Одеса,Таїрова': 'Cinema 3O',
    'NEW Планета Кіно Одеса, Котовського': 'Cinema 4O',
    'NEW Планета Кіно Суми': 'Cinema 5S',
    'NEW Планета-Кіно IMAX Львів': 'Cinema 6L',
    'NEW Планета-Кіно IMAX Харків': 'Cinema 7H',
    'Київ (River Mall)': 'Cinema 8K',
    'NEW Блокбастер-Кино': 'Cinema 9K'

})

In [293]:
df['hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

In [294]:
def time_period(hour):
    if 6 <= hour < 12:
        return 'Ранок'
    elif 12 <= hour < 18:
        return 'День'
    elif 18 <= hour < 24:
        return 'Вечір'
    else:
        return 'Ніч'

df['period'] = df['hour'].apply(time_period)
period_counts = df['period'].value_counts()
period_percent = (period_counts / period_counts.sum()) * 100

print("Розподіл за періодами доби:", '\n', period_percent)

Розподіл за періодами доби: 
 День     44.359214
Вечір    38.937622
Ранок    16.368503
Ніч       0.334661
Name: period, dtype: float64


In [ ]:
df[['budget', 'revenue']] = df[['budget', 'revenue']].replace(0, np.nan)

In [297]:
df['occupacyRate'] = df['quantitySold']/df['quantityPlan']
df['profitRatio'] = df['profitEarned']/df['profitPlan']

Nan fill in cat_cols

In [298]:
cat_cols = ['Distributor', 'Studio', 'genres', 'directors', 'productionCountries', 'mpaaRating', 'originalLanguage']
for col in cat_cols:
    mode = df[col].mode(dropna=True)
    if mode.empty or mode is None :
        continue
    top = mode.iloc[0]
    if df[col].dtype.name == "category" and top not in df[col].cat.categories:
        df[col] = df[col].cat.add_categories([top])
    df[col].fillna(top, inplace=True)


In [185]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 234200 entries, 0 to 234265
Data columns (total 35 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Cinema               234200 non-null  object 
 1   Film                 234200 non-null  object 
 2   durationMin          234200 non-null  int64  
 3   Distributor          234200 non-null  object 
 4   Studio               234200 non-null  object 
 5   Auditorium           234200 non-null  object 
 6   Technology           234200 non-null  object 
 7   Number               234200 non-null  int64  
 8   sessionCanceled      234200 non-null  object 
 9   isPremiere           234200 non-null  object 
 10  Date                 234200 non-null  object 
 11  Time                 234200 non-null  object 
 12  seatType             234200 non-null  object 
 13  quantitySold         234200 non-null  int64  
 14  quantityPlan         234200 non-null  int64  
 15  quantityUnsold   

In [301]:
df.to_excel('backup.xlsx', index=False)